# PFAS × ACS — 3D Mapping (pydeck)

Builds an interactive choropleth (shaded by an ACS variable) + 3D bars (PFAS reading) for a
chosen **month**, and writes a standalone HTML.

**Why a prep step:** the raw TIGER ZCTA shapefile is ~500 MB of full-resolution boundaries for
all ~33k ZCTAs. We never carry that into the repo or the HTML. Step 0 runs **once**: filter to
the ZCTAs we actually have data for, simplify, and save a small GeoParquet. That small file
(a few MB) is all the map needs — keeping both the repo and the exported HTML light.

In [49]:
import os, json, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import pydeck as pdk
import geopandas as gpd


DATA  = Path(os.environ.get("DATA",  "../data/processed/pfas_acs_zcta_month.parquet"))
TIGER = Path(os.environ.get("TIGER", "../data/raw/tiger/cb_2020_us_zcta520_500k.shp"))  # raw download (gitignored)
GEOM  = Path(os.environ.get("GEOM",  "../data/processed/zcta_simplified.parquet"))      # small, committed
OUTLINE = Path(os.environ.get("OUTLINE", "../data/processed/zcta_outline_all.parquet"))
OUTDIR= Path(os.environ.get("OUTDIR","../maps")); OUTDIR.mkdir(parents=True, exist_ok=True)

## 0. One-time geometry prep (run once)

Reads the big shapefile, keeps only ZCTAs present in your data, reprojects to WGS84, simplifies,
and saves `zcta_simplified.parquet`. Skips automatically if that file already exists.

**Memory:** download the **cartographic boundary** file (`cb_2020_us_zcta520_500k`) rather than the
full `tl_2020_us_zcta520` — it's the pre-generalized version and far lighter to read. If even that
strains Colab, switch to the High-RAM runtime for this one cell; afterwards you only ever load the
small GeoParquet.

In [57]:
if GEOM.exists():
    print("geometry already prepared:", GEOM)
else:
    keep = set(pd.read_parquet(DATA, columns=["zcta"])["zcta"].astype(str).str.zfill(5))
    g = gpd.read_file(TIGER)                                   # uses pyogrio (fast, lighter)
    zcol = next(c for c in g.columns if c.upper().startswith("ZCTA5CE"))
    g = g[[zcol, "geometry"]].rename(columns={zcol: "ZCTA"})
    g["ZCTA"] = g["ZCTA"].astype(str).str.zfill(5)
    g = g[g["ZCTA"].isin(keep)].to_crs(4326)                   # filter FIRST, then reproject
    g["geometry"] = g.simplify(0.03, preserve_topology=True) 
    g.to_parquet(GEOM)
    print(f"wrote {GEOM}  ({GEOM.stat().st_size/1e6:.1f} MB, {len(g):,} ZCTAs)")

geometry already prepared: ..\data\processed\zcta_simplified.parquet


## 1. Load data + geometry, merge

In [58]:
df  = pd.read_parquet(DATA)
df["zcta"] = df["zcta"].astype(str).str.zfill(5)

geo = gpd.read_parquet(GEOM)
geo["ZCTA"] = geo["ZCTA"].astype(str).str.zfill(5)
geo = geo.rename(columns={"ZCTA": "zcta"})[["zcta", "geometry"]]   # avoid clash w/ df's ACS 'ZCTA'

gdf = geo.merge(df, on="zcta", how="inner")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=geo.crs)
# centroids for the 3D bars (projected CRS -> accurate, no warning), back to lon/lat
cent = gdf.to_crs(5070).geometry.centroid.to_crs(4326)
gdf["lon"], gdf["lat"] = cent.x.values, cent.y.values

ACS_VARS = [c for c in ["total_pop","median_hh_income","pct_below_poverty","pct_nonwhite",
                        "pct_hispanic","pct_black_nh","pct_asian_nh"] if c in gdf.columns]
months = sorted(gdf["month"].dropna().unique())
print(f"{gdf['zcta'].nunique():,} ZCTAs with geometry | months: {len(months)} "
      f"({pd.Timestamp(months[0]):%Y-%m} … {pd.Timestamp(months[-1]):%Y-%m})")
print("available ACS variables:", ACS_VARS)

8,709 ZCTAs with geometry | months: 84 (2013-01 … 2025-12)
available ACS variables: ['total_pop', 'median_hh_income', 'pct_below_poverty', 'pct_nonwhite', 'pct_hispanic', 'pct_black_nh', 'pct_asian_nh']


## 2. Blue → orange color ramp

In [59]:
BLUE, MID, ORANGE = np.array([74,125,224.]), np.array([150,150,170.]), np.array([224,114,74.])
def ramp(t):
    t = np.clip(np.nan_to_num(np.asarray(t, float), nan=0.0), 0, 1)
    lo = t < 0.5
    out = np.empty((t.size, 3))
    out[lo]  = BLUE + (MID-BLUE)   * (t[lo]/0.5)[:,None]
    out[~lo] = MID  + (ORANGE-MID) * ((t[~lo]-0.5)/0.5)[:,None]
    return out.round().astype(int)

def norm(s, lo_q=0.05, hi_q=0.95):
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.quantile(lo_q), s.quantile(hi_q)
    hi = hi if hi > lo else lo + 1
    return ((s - lo) / (hi - lo)).clip(0, 1)

## 3. The map function

`make_map(month, acs_var)` → choropleth shaded by `acs_var` + 3D bars whose height **and** color
encode the mean PFAS reading, both on the blue→orange scale. Returns the pydeck `Deck` and writes
a standalone HTML.

> **Note**: Month parameter is left as future work. Currently it will take up too much memory.

In [82]:
ACS_LABELS = {
    "total_pop": "Total population", "median_hh_income": "Median household income ($)",
    "pct_below_poverty": "% below poverty", "pct_nonwhite": "% people of color",
    "pct_hispanic": "% Hispanic", "pct_black_nh": "% Black (NH)", "pct_asian_nh": "% Asian (NH)",
}

def make_map(month, acs_var, gdf=gdf, elevation_max=180000, radius=1300, alpha=185,
             n_hotspots=12, layers="both", save=True):
    if acs_var not in gdf.columns:
        raise ValueError(f"{acs_var!r} not in data. Options: {ACS_VARS}")
    if layers not in ("both", "choropleth", "bars"):
        raise ValueError("layers must be 'both', 'choropleth', or 'bars'")

    def _union_names(s):   # combine "; "-joined PWS names across a ZCTA's months
        out = set()
        for v in s.dropna():
            out.update(p.strip() for p in str(v).split(";") if p.strip())
        return "; ".join(sorted(out))

    # snapshot: all-time (one row per ZCTA) or a single month
    if month is None:
        ak = {"mean_result_ngL": ("mean_result_ngL", "mean"), acs_var: (acs_var, "first")}
        for col in ("city", "state"):
            if col in gdf.columns:
                ak[col] = (col, "first")
        if "pws_names" in gdf.columns:
            ak["pws_names"] = ("pws_names", _union_names)
        agg = gdf.groupby("zcta").agg(**ak).reset_index()
        base = gdf.drop_duplicates("zcta")[["zcta", "geometry", "lon", "lat"]]
        snap = gpd.GeoDataFrame(base.merge(agg, on="zcta"), geometry="geometry", crs=gdf.crs)
        label = "all-time"
    else:
        snap = gdf[gdf["month"] == pd.Timestamp(month)].copy()
        if snap.empty:
            raise ValueError(f"no rows for month {month}")
        label = pd.Timestamp(month).strftime("%Y-%m")
    dem_label = ACS_LABELS.get(acs_var, acs_var)

    # choropleth shaded by the demographic variable
    dcol = ramp(norm(snap[acs_var]))
    snap["fill_color"] = [[int(r), int(g), int(b), alpha] for r, g, b in dcol]

    # per-feature tooltip: ZCTA — City, ST / demographic / PFAS / systems
    def fmt(v):
        return "n/a" if pd.isna(v) else (f"{v:,.1f}" if isinstance(v, (float, np.floating)) else f"{v}")
    def loc(z, c, st):
        cs = ", ".join([x for x in [c if isinstance(c, str) and c else None,
                                    st if isinstance(st, str) and st else None] if x])
        return f"ZCTA {z} — {cs}" if cs else f"ZCTA {z}"
    def pwsline(p):
        if not isinstance(p, str) or not p:
            return ""
        parts = [x.strip() for x in p.split(";") if x.strip()]
        return "\nSystems: " + "; ".join(parts[:3]) + (f" +{len(parts)-3} more" if len(parts) > 3 else "")
    csv = snap["city"] if "city" in snap.columns else [None] * len(snap)
    ssv = snap["state"] if "state" in snap.columns else [None] * len(snap)
    psv = snap["pws_names"] if "pws_names" in snap.columns else [None] * len(snap)
    snap["tip"] = [loc(z, c, st) + f"\n{dem_label}: {fmt(d)}\nPFAS mean: {fmt(p)} ng/L" + pwsline(pw)
                   for z, c, st, d, p, pw in
                   zip(snap["zcta"], csv, ssv, snap[acs_var], snap["mean_result_ngL"], psv)]

    choropleth = pdk.Layer(
        "GeoJsonLayer", json.loads(snap[["zcta", "tip", "fill_color", "geometry"]].to_json()),
        get_fill_color="properties.fill_color", get_line_color=[60, 60, 70], auto_highlight=True,
        line_width_min_pixels=0.2, stroked=True, filled=True, extruded=False, pickable=True)

    # thin 3D columns from PFAS
    bars = snap.dropna(subset=["mean_result_ngL"]).copy()
    pn = norm(bars["mean_result_ngL"], hi_q=0.99)
    bcol = ramp(pn)
    bars["bar_color"] = [[int(r), int(g), int(b), 230] for r, g, b in bcol]
    bars["elev"] = (pn * elevation_max).fillna(0)
    columns = pdk.Layer(
        "ColumnLayer", bars[["lon", "lat", "elev", "bar_color", "tip", "zcta", "mean_result_ngL"]],
        get_position=["lon", "lat"], get_elevation="elev", elevation_scale=1,
        radius=radius, get_fill_color="bar_color", pickable=True, auto_highlight=True)

    # red ring + light-pink fill over top EJ-coincidence ZCTAs — ALWAYS shown
    bars["score"] = norm(bars[acs_var]).fillna(0).values * pn.values
    hs = bars.nlargest(n_hotspots, "score")
    halos = [pdk.Layer(
        "ScatterplotLayer", hs[["lon", "lat"]], get_position=["lon", "lat"],
        filled=True, stroked=True,
        get_fill_color=[255, 190, 200, 210],
        get_line_color=[210, 20, 20, 230], line_width_min_pixels=2.5,
        radius_min_pixels=16, radius_max_pixels=16, pickable=False)]

    # assemble layers per mode; halos are always added last (on top)
    active = []
    if layers in ("both", "choropleth"):
        active.append(choropleth)
    if layers in ("both", "bars"):
        active.append(columns)
    active += halos

    view = pdk.ViewState(latitude=39.5, longitude=-98.5, zoom=4.6, pitch=86, bearing=0)
    map_view = pdk.View("MapView", controller={"dragRotate": True, "touchRotate": True})
    deck = pdk.Deck(layers=active, initial_view_state=view, views=[map_view],
                    map_provider="carto", map_style="light",
                    tooltip={"text": "{tip}"})

    if save:
        out = OUTDIR / f"map_{label}_{acs_var}_{layers}.html"
        deck.to_html(str(out), notebook_display=False)
        print(f"{label} [{layers}]: {len(active)} layers -> {out.name} "
              f"({out.stat().st_size/1e6:.1f} MB)")
    return deck

## 4. Populating All Map Configurations

In [ ]:
for var in ACS_VARS:
    make_map(month=None, acs_var=var, layers="both")
    make_map(month=None, acs_var=var, layers="choropleth")

make_map(month=None, acs_var=ACS_VARS[0], layers="bars") 

all-time [both]: 3 layers -> map_all-time_total_pop_both.html (16.2 MB)
all-time [choropleth]: 2 layers -> map_all-time_total_pop_choropleth.html (12.2 MB)
all-time [both]: 3 layers -> map_all-time_median_hh_income_both.html (16.4 MB)
all-time [choropleth]: 2 layers -> map_all-time_median_hh_income_choropleth.html (12.3 MB)
all-time [both]: 3 layers -> map_all-time_pct_below_poverty_both.html (16.1 MB)
all-time [choropleth]: 2 layers -> map_all-time_pct_below_poverty_choropleth.html (12.2 MB)
all-time [both]: 3 layers -> map_all-time_pct_nonwhite_both.html (16.2 MB)


## Web integration & repo notes

- **One month per HTML:** because the function renders a single snapshot, each file is a few MB,
  not 1 GB. Pre-generate the snapshots you want (loop over `months` / `ACS_VARS`) into `maps/`.
- **Embed in the site** via the iframe placeholder: `src="maps/map_2024-04_pct_nonwhite.html"`.
- **`.gitignore`** the raw shapefile folder (`data/raw/tiger/`); commit only `zcta_simplified.parquet`
  and the small per-snapshot HTMLs.
- If files are still too big, raise the `simplify` tolerance (step 0) and/or drop coordinate
  precision; that shrinks both the GeoParquet and every HTML.